# PubMed pathogen/zoonosis corpus screening

This notebook collects recall-oriented animal-infection and zoonosis corpora,
then uses a pathogen-specific LLM screen to validate attribution and derive
corpus-relative categories 1, 2, and 3. All search and screening outputs are
local, resumable Parquet checkpoints under `outputs/pubmed_screening/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    OpenAIChatCompleter,
    PathogenSpec,
    PubMedClient,
    build_pathogen_search_query,
    collect_pubmed_corpus,
    derive_category_counts,
    load_search_bundle,
    screen_pubmed_corpus,
)

## Configure pathogens, YAML search bundles, and checkpoints

In [ ]:
PUBMED_EMAIL = os.environ.get('PUBMED_EMAIL', '')
if not PUBMED_EMAIL:
    raise RuntimeError('Set PUBMED_EMAIL before calling the NCBI E-utilities.')

PATHOGENS = {
    'Nipah virus': ['Nipah', 'NiV'],
}
SEARCH_BUNDLE_DIR = PROJECT_ROOT / 'assets' / 'search_bundles'
SEARCH_BUNDLES = {
    bundle.search_type: bundle
    for bundle in (
        load_search_bundle(SEARCH_BUNDLE_DIR / 'pubmed_animal_evidence.yaml', expected_search_type='animal'),
        load_search_bundle(SEARCH_BUNDLE_DIR / 'pubmed_zoonosis_evidence.yaml', expected_search_type='zoonosis'),
    )
}

PUBMED_MAX_RESULTS_PER_QUERY = 5000
PUBMED_SEARCH_PAGE_SIZE = 1000
PUBMED_FETCH_BATCH_SIZE = 200
RESUME = True
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'pubmed_screening'

OPENAI_MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
LLM_MAX_TOKENS = 1024
LLM_RETRIES = 3
LLM_MAX_CALLS = None
LLM_SAVE_EVERY = 10
RETRY_FAILED_LLM_ROWS = False

pubmed = PubMedClient(
    email=PUBMED_EMAIL,
    api_key=os.environ.get('PUBMED_API_KEY'),
    tool='recursive-framing-graphicalizer',
    retries=3,
)


## Preview the generated queries

In [ ]:
for pathogen, aliases in PATHOGENS.items():
    spec = PathogenSpec(pathogen, tuple(aliases))
    print(f'--- {pathogen} / animal ---')
    print(build_pathogen_search_query(spec, 'animal', search_bundles=SEARCH_BUNDLES))
    print(f'--- {pathogen} / zoonosis ---')
    print(build_pathogen_search_query(spec, 'zoonosis', search_bundles=SEARCH_BUNDLES))

## Collect and resume the deduplicated corpus

In [ ]:
collection = collect_pubmed_corpus(
    pubmed,
    PATHOGENS,
    OUTPUT_DIR,
    search_bundles=SEARCH_BUNDLES,
    max_results_per_query=PUBMED_MAX_RESULTS_PER_QUERY,
    search_page_size=PUBMED_SEARCH_PAGE_SIZE,
    fetch_batch_size=PUBMED_FETCH_BATCH_SIZE,
    resume=RESUME,
)
corpus = collection.corpus
print('Corpus rows:', len(corpus))
print('Search failures:', collection.manifest['search_failures'])
print(corpus.groupby(['pathogen', 'fetch_status']).size())

## Validate every candidate abstract with the LLM

In [ ]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the LLM screen.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
screening_run = screen_pubmed_corpus(
    corpus,
    PATHOGENS,
    llm,
    OUTPUT_DIR,
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Screened rows:', len(screening_run.screening))
print('Retryable failures:', len(screening_run.failures))

## Derive corpus-relative categories and counts

In [ ]:
screened_with_categories, zoonosis_timeline, category_counts = derive_category_counts(
    screening_run.screening,
    OUTPUT_DIR,
)
print('First confirmed zoonosis by pathogen:')
display(zoonosis_timeline)
print('Category counts:')
display(category_counts.sort_values(['pathogen', 'publication_year', 'category'], na_position='first'))
print('Review-required rows:', int(screened_with_categories['review_required'].fillna(True).sum()))

## Inspect quality and provenance

Category 1 means no confirmed zoonosis evidence was found in this searched corpus; it is not proof that the pathogen has never undergone zoonosis. Review-required and failed rows are excluded from final counts.

In [ ]:
print('Output directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
display(
    screened_with_categories[[
        'pathogen', 'pmid', 'llm_category', 'final_category',
        'target_pathogen_supported', 'animal_infection_supported',
        'zoonosis_supported', 'confidence', 'review_required', 'rationale'
    ]].head(20)
)